#  CineMatch — Concept Notebook
### How a Content-Based Movie Recommender Works, From Scratch

---

This notebook is the conceptual backbone of the **CineMatch** project.  
It strips away the FastAPI server, the SQLite database, the user-auth system, and the poster resolver — and focuses entirely on the **core recommender engine**.

By the end of this notebook you will have:
- Loaded and merged the TMDB 5000 movies dataset
- Engineered a rich `tags` feature per movie
- Built a TF-IDF vector space
- Computed a cosine-similarity matrix
- Written four recommendation strategies used in production

**Dataset required:** Download from Kaggle — [TMDB 5000 Movie Dataset](https://www.kaggle.com/datasets/tmdb/tmdb-movie-metadata)  

---

##  Architecture Overview

```
Raw CSVs  ──►  Merge & Clean  ──►  Feature Engineering  ──►  TF-IDF Matrix
                                                                    │
                                                                    ▼
User Input  ──────────────────────────────────────────►  Cosine Similarity
  (watched movies / preferred genres)                        │
                                                             ▼
                                                    Ranked Recommendations
```



---
## Part 1 — Setup & Imports

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Drive already mounted at /content/drive; to attempt to forcibly remount, call drive.mount("/content/drive", force_remount=True).


In [3]:
# Core data science stack
import ast
import pickle
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
from sklearn.feature_extraction.text import TfidfVectorizer
from sklearn.metrics.pairwise import cosine_similarity

warnings.filterwarnings("ignore")
pd.set_option("display.max_colwidth", 60)

DATA_DIR  = Path("data")
MODEL_DIR = Path("models")
MODEL_DIR.mkdir(exist_ok=True)

print("Imports OK")

Imports OK


---
## Part 2 — Load & Merge the Raw Data

The TMDB dataset comes as **two** CSV files that must be joined on `id`:

| File | Key columns |
|------|-------------|
| `tmdb_5000_movies.csv` | id, title, overview, genres, keywords, vote_average, vote_count, release_date, runtime, popularity |
| `tmdb_5000_credits.csv` | movie_id (→ id), cast, crew |

In [4]:
movies  = pd.read_csv(DATA_DIR / "/content/drive/MyDrive/Python_ML/ml_projects/movie_recommender/tmdb_5000_movies.csv")
credits = pd.read_csv(DATA_DIR / "/content/drive/MyDrive/Python_ML/ml_projects/movie_recommender/tmdb_5000_credits.csv")

# credits uses 'movie_id' — rename to match movies' 'id'
if "movie_id" in credits.columns:
    credits.rename(columns={"movie_id": "id"}, inplace=True)

df = movies.merge(credits, on="id")
df.rename(columns={"title_x": "title"}, inplace=True)

print(f"Rows after merge : {len(df):,}")
print(f"Columns          : {list(df.columns)}")

Rows after merge : 4,803
Columns          : ['budget', 'genres', 'homepage', 'id', 'keywords', 'original_language', 'original_title', 'overview', 'popularity', 'production_companies', 'production_countries', 'release_date', 'revenue', 'runtime', 'spoken_languages', 'status', 'tagline', 'title', 'vote_average', 'vote_count', 'title_y', 'cast', 'crew']


In [5]:
# Keep only the columns we actually need
KEEP = ["id", "title", "overview", "genres", "keywords",
        "cast", "crew", "vote_average", "vote_count",
        "release_date", "runtime", "popularity"]

df = df[[c for c in KEEP if c in df.columns]].copy()
df.dropna(subset=["overview", "genres"], inplace=True)
df.reset_index(drop=True, inplace=True)

print(f"Clean rows : {len(df):,}")
df.head(3)

Clean rows : 4,800


,id,title,overview,genres,keywords,cast,crew,vote_average,vote_count,release_date,runtime,popularity
0,19995,Avatar,"In the 22nd century, a paraplegic Marine is dispatched t...","[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""name"": ""Adven...","[{""id"": 1463, ""name"": ""culture clash""}, {""id"": 2964, ""na...","[{""cast_id"": 242, ""character"": ""Jake Sully"", ""credit_id""...","[{""credit_id"": ""52fe48009251416c750aca23"", ""department"":...",7.2,11800,2009-12-10,162.0,150.437577
1,285,Pirates of the Caribbean: At World's End,"Captain Barbossa, long believed to be dead, has come bac...","[{""id"": 12, ""name"": ""Adventure""}, {""id"": 14, ""name"": ""Fa...","[{""id"": 270, ""name"": ""ocean""}, {""id"": 726, ""name"": ""drug...","[{""cast_id"": 4, ""character"": ""Captain Jack Sparrow"", ""cr...","[{""credit_id"": ""52fe4232c3a36847f800b579"", ""department"":...",6.9,4500,2007-05-19,169.0,139.082615
2,206647,Spectre,A cryptic message from Bond’s past sends him on a trail ...,"[{""id"": 28, ""name"": ""Action""}, {""id"": 12, ""name"": ""Adven...","[{""id"": 470, ""name"": ""spy""}, {""id"": 818, ""name"": ""based ...","[{""cast_id"": 1, ""character"": ""James Bond"", ""credit_id"": ...","[{""credit_id"": ""54805967c3a36829b5002c41"", ""department"":...",6.3,4466,2015-10-26,148.0,107.376788


---
## Part 3 — Parse JSON-like Columns

Columns like `genres`, `keywords`, `cast`, and `crew` are stored as **JSON strings** inside the CSV.  
Each looks like: `"[{'id': 28, 'name': 'Action'}, {'id': 12, 'name': 'Adventure'}]"`

We use `ast.literal_eval` to safely convert them to Python lists.

In [6]:
def safe_eval(val):
    """Safely parse a JSON-like string into a Python object."""
    try:
        return ast.literal_eval(val)
    except (ValueError, SyntaxError):
        return []


def extract_names(obj_list, limit=3):
    """Pull the 'name' field from the first `limit` items."""
    if isinstance(obj_list, list):
        return [item["name"] for item in obj_list[:limit] if "name" in item]
    return []


def extract_director(crew_list):
    """Find the first person with job == 'Director'."""
    if isinstance(crew_list, list):
        for member in crew_list:
            if member.get("job") == "Director":
                return [member["name"]]
    return []


def collapse_spaces(names):
    """Turn 'Sam Raimi' → 'SamRaimi' so TF-IDF treats it as one token."""
    return [n.replace(" ", "") for n in names]


# Parse all four JSON columns
for col in ["genres", "keywords", "cast", "crew"]:
    df[col] = df[col].apply(safe_eval)

df["genres_list"]   = df["genres"].apply(lambda x: extract_names(x, 5))
df["keywords_list"] = df["keywords"].apply(lambda x: extract_names(x, 10))
df["cast_list"]     = df["cast"].apply(lambda x: collapse_spaces(extract_names(x, 3)))
df["director"]      = df["crew"].apply(lambda x: collapse_spaces(extract_director(x)))

# Preview
df[["title", "genres_list", "cast_list", "director"]].head(4)

,title,genres_list,cast_list,director
0,Avatar,"[Action, Adventure, Fantasy, Science Fiction]","[SamWorthington, ZoeSaldana, SigourneyWeaver]",[JamesCameron]
1,Pirates of the Caribbean: At World's End,"[Adventure, Fantasy, Action]","[JohnnyDepp, OrlandoBloom, KeiraKnightley]",[GoreVerbinski]
2,Spectre,"[Action, Adventure, Crime]","[DanielCraig, ChristophWaltz, LéaSeydoux]",[SamMendes]
3,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]","[ChristianBale, MichaelCaine, GaryOldman]",[ChristopherNolan]


---
## Part 4 — Feature Engineering: The `tags` Column

This is the **heart of the recommender**.

We smash all meaningful signals into a single string of space-separated tokens per movie:

| Signal | Weight trick |
|--------|-------------|
| genres | repeated ×2 to boost genre matching |
| keywords | taken as-is |
| top-3 cast | spaces collapsed so "Tom Hanks" → "TomHanks" (one token) |
| director | same space-collapse trick |
| overview | first 50 words, lowercased |

Everything goes **lowercase** before vectorisation.

In [7]:
def build_tags(row):
    overview_tokens = (
        row["overview"].lower().split()[:50]
        if isinstance(row["overview"], str) else []
    )
    parts = (
        row["genres_list"]   * 2   +   # genres weighted double
        row["keywords_list"]       +
        row["cast_list"]           +
        row["director"]            +
        overview_tokens
    )
    return " ".join(parts).lower()


df["tags"] = df.apply(build_tags, axis=1)

# Inspect a sample
sample = df.loc[df["title"] == "The Dark Knight", "tags"].values
if len(sample):
    print("Tags for 'The Dark Knight':")
    print(sample[0][:300], "...")
else:
    print(df["tags"].iloc[0][:300], "...")

Tags for 'The Dark Knight':
drama action crime thriller drama action crime thriller dc comics crime fighter secret identity scarecrow sadism chaos gotham city vigilante joker superhero christianbale heathledger aaroneckhart christophernolan batman raises the stakes in his war on crime. with the help of lt. jim gordon and distr ...


---
## Part 5 — TF-IDF Vectorisation

**TF-IDF** (Term Frequency–Inverse Document Frequency) converts each movie's `tags` string into a sparse numeric vector.

- **TF**: how often a word appears in *this* movie's tags
- **IDF**: penalises words that appear in *every* movie (e.g., "the", common genres) — making rarer, more specific tokens more influential

**Key hyper-parameters used in CineMatch:**

| Parameter | Value | Reason |
|-----------|-------|--------|
| `max_features` | 15 000 | caps vocabulary size |
| `stop_words` | `"english"` | removes noise words |
| `ngram_range` | `(1, 2)` | captures bi-grams like "space shuttle" |
| `min_df` | 2 | drops tokens that only appear once (typos / noise) |

In [8]:
tfidf = TfidfVectorizer(
    max_features=15_000,
    stop_words="english",
    ngram_range=(1, 2),
    min_df=2,
)

tfidf_matrix = tfidf.fit_transform(df["tags"])

print(f"Matrix shape : {tfidf_matrix.shape}")
print(f"  → {tfidf_matrix.shape[0]} movies × {tfidf_matrix.shape[1]} token features")
print(f"  → Sparsity: {100 * (1 - tfidf_matrix.nnz / (tfidf_matrix.shape[0] * tfidf_matrix.shape[1])):.1f}% zeros")

Matrix shape : (4800, 15000)
  → 4800 movies × 15000 token features
  → Sparsity: 99.8% zeros


---
## Part 6 — Cosine Similarity Matrix

With every movie as a vector in a 15,000-dimensional space, **cosine similarity** measures the angle between two vectors.  
- Score = **1.0** → identical direction → very similar movies  
- Score = **0.0** → perpendicular → nothing in common  

$$
\text{similarity}(A, B) = \frac{A \cdot B}{\|A\| \cdot \|B\|}
$$

The result is a **N × N matrix** (N = number of movies). This is the most expensive step — roughly 30–90 seconds for 4,800 movies.  
In production, CineMatch saves it as `cosine_sim.pkl` so the server loads a pre-computed result at startup.

In [20]:
print("Computing cosine similarity matrix...")
cosine_sim = cosine_similarity(tfidf_matrix, tfidf_matrix)

print(f"Matrix shape : {cosine_sim.shape}")
print(f"Memory usage : {cosine_sim.nbytes / 1024**2:.0f} MB")
print("\nDone ")

Computing cosine similarity matrix...
Matrix shape : (4800, 4800)
Memory usage : 176 MB

Done 


---
## Part 7 — Build Lookup Indexes

We need fast mappings to look up a movie by title or TMDB id:

In [10]:
# title  → row index in df
title_to_idx = pd.Series(df.index, index=df["title"]).drop_duplicates()

# TMDB id → row index in df
id_to_idx = pd.Series(df.index, index=df["id"]).drop_duplicates()

# lowercase normalised title for search
df["norm_title"] = df["title"].str.lower().str.strip()

print(f"Indexed {len(title_to_idx):,} unique titles")
print(f"Sample lookup — 'Avatar' → row {title_to_idx.get('Avatar', 'NOT FOUND')}")

Indexed 4,800 unique titles
Sample lookup — 'Avatar' → row 0


---
## Part 8 — Recommendation Strategies

CineMatch ships **four strategies**. Let's implement them all.

### Helper: format a result row

In [11]:
def fmt(row, score=None, score_key="score"):
    """Convert a DataFrame row to a tidy result dict."""
    release_date = str(row.get("release_date", "") or "")
    result = {
        "title"       : row["title"],
        "genres"      : row.get("genres_list", []),
        "vote_avg"    : round(float(row.get("vote_average", 0) or 0), 1),
        "year"        : release_date[:4] if release_date[:4].isdigit() else "?",
    }
    if score is not None:
        result[score_key] = round(float(score), 4)
    return result


def show(movies_list, n=10):
    """Pretty-print a list of result dicts."""
    return pd.DataFrame(movies_list[:n])

### Strategy 1 — Content-Based (by single movie title)

Given one seed movie, rank all others by their cosine similarity to it.  
This is the classic "if you liked X, you'll love Y" pattern.

In [12]:
def recommend_by_title(title, top_n=10):
    """Content-based recommendations for a single seed movie."""
    if title not in title_to_idx:
        print(f"'{title}' not found.")
        return []

    idx = title_to_idx[title]

    # Row idx from the cosine matrix gives similarity to every other movie
    sim_scores = list(enumerate(cosine_sim[idx]))

    # Sort descending, skip index 0 (the movie itself)
    sim_scores = sorted(sim_scores, key=lambda x: x[1], reverse=True)[1:top_n + 1]

    return [fmt(df.iloc[i], score=s, score_key="similarity") for i, s in sim_scores]


# ── Demo ──────────────────────────────────────────────────────────────────────
print("Recommendations for: The Dark Knight")
show(recommend_by_title("The Dark Knight"))

Recommendations for: The Dark Knight


,title,genres,vote_avg,year,similarity
0,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]",7.6,2012,0.5133
1,Batman Begins,"[Action, Crime, Drama]",7.5,2005,0.3987
2,Batman Forever,"[Action, Crime, Fantasy]",5.2,1995,0.3580
3,Batman Returns,"[Action, Fantasy]",6.6,1992,0.3248
4,Batman,"[Fantasy, Action]",7.0,1989,0.2932
5,Amidst the Devil's Wings,"[Drama, Action, Crime]",0.0,2014,0.2930
6,Batman & Robin,"[Action, Crime, Fantasy]",4.2,1997,0.2621
7,Chicago Overcoat,"[Action, Crime, Thriller]",6.1,2009,0.2196
8,Law Abiding Citizen,"[Drama, Crime, Thriller]",7.2,2009,0.2145
9,Exiled,"[Action, Crime, Thriller]",7.0,2006,0.2111


### Strategy 2 — History-Based (multiple seed movies)

When a user has a **watch history** we average the similarity vectors across all their seen movies to build a taste *profile*, then rank unseen films against that profile.

```
profile = mean( cosine_sim[movie_1], cosine_sim[movie_2], ... )
```

In [13]:
def recommend_by_history(watched_titles, top_n=10):
    """Recommend based on a list of movies the user has already seen."""
    valid_idxs = [title_to_idx[t] for t in watched_titles if t in title_to_idx]
    if not valid_idxs:
        print("None of the provided titles were found.")
        return []

    # Build a taste profile: average similarity row across all watched movies
    profile = np.zeros(len(df), dtype=np.float64)
    for idx in valid_idxs:
        profile += cosine_sim[idx]
    profile /= len(valid_idxs)

    watched_set = set(valid_idxs)
    ranked = sorted(enumerate(profile), key=lambda x: x[1], reverse=True)

    results = []
    for movie_idx, score in ranked:
        if movie_idx in watched_set:   # skip already-seen
            continue
        results.append(fmt(df.iloc[movie_idx], score=score, score_key="profile_score"))
        if len(results) >= top_n:
            break
    return results


# ── Demo ──────────────────────────────────────────────────────────────────────
history = ["The Dark Knight", "Inception", "Interstellar"]
print(f"History: {history}")
show(recommend_by_history(history))

History: ['The Dark Knight', 'Inception', 'Interstellar']


,title,genres,vote_avg,year,profile_score
0,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]",7.6,2012,0.1881
1,Batman Begins,"[Action, Crime, Drama]",7.5,2005,0.1577
2,2001: A Space Odyssey,"[Science Fiction, Mystery, Adventure]",7.9,1968,0.1371
3,Midnight Special,"[Adventure, Drama, Science Fiction]",6.2,2016,0.1306
4,Gattaca,"[Thriller, Science Fiction, Mystery, Romance]",7.5,1997,0.1294
5,Timecop,"[Thriller, Science Fiction, Action, Crime]",5.5,1994,0.1272
6,Contact,"[Drama, Science Fiction, Mystery]",7.2,1997,0.1272
7,Looper,"[Action, Thriller, Science Fiction]",6.6,2012,0.1225
8,Silent Running,"[Adventure, Drama, Science Fiction]",6.3,1972,0.1222
9,Batman Forever,"[Action, Crime, Fantasy]",5.2,1995,0.1221


### Strategy 3 — Genre-Based (with Bayesian score)

Filter to movies that share at least one genre with the user's preferences, then rank by a **Bayesian average** instead of raw vote_average.

Why Bayesian? A movie with 8.5 rating from **50 votes** should rank lower than a 8.0 from **50,000 votes**.  
The Bayesian formula shrinks low-vote-count ratings toward the global mean:

$$
\text{score} = \frac{v}{v + m} \cdot R + \frac{m}{v + m} \cdot C
$$

- **v** = vote count for this movie  
- **m** = minimum votes threshold (floor)  
- **R** = this movie's average rating  
- **C** = global average rating across all movies

In [14]:
GLOBAL_MEAN = float(df["vote_average"].mean())
MIN_VOTES   = 100  # floor for Bayesian credibility

def bayesian_score(row, floor=MIN_VOTES, global_mean=GLOBAL_MEAN):
    v = row["vote_count"]
    R = row["vote_average"]
    return (v / (v + floor)) * R + (floor / (v + floor)) * global_mean


def recommend_by_genres(genre_list, top_n=10, min_rating=6.0):
    """Filter by genres, rank by Bayesian score."""
    target_genres = {g.strip().title() for g in genre_list}

    def has_genre(g_list):
        return isinstance(g_list, list) and any(g in target_genres for g in g_list)

    filtered = df[
        df["genres_list"].apply(has_genre) &
        (df["vote_average"] >= min_rating)
    ].copy()

    if filtered.empty:
        print(f"No results for genres: {genre_list}")
        return []

    filtered["_score"] = filtered.apply(bayesian_score, axis=1)
    top = filtered.sort_values("_score", ascending=False).head(top_n)

    return [fmt(row, score=row["_score"], score_key="bayesian_score") for _, row in top.iterrows()]


# ── Demo ──────────────────────────────────────────────────────────────────────
print("Top Sci-Fi + Thriller movies by Bayesian score:")
show(recommend_by_genres(["Science Fiction", "Thriller"]))

Top Sci-Fi + Thriller movies by Bayesian score:


,title,genres,vote_avg,year,bayesian_score
0,Pulp Fiction,"[Thriller, Crime]",8.3,1994,8.2741
1,The Dark Knight,"[Drama, Action, Crime, Thriller]",8.2,2008,8.1826
2,The Empire Strikes Back,"[Adventure, Action, Science Fiction]",8.2,1980,8.1648
3,Psycho,"[Drama, Horror, Thriller]",8.2,1960,8.1129
4,Inception,"[Action, Thriller, Science Fiction, Mystery, Adventure]",8.1,2010,8.0855
5,Interstellar,"[Adventure, Drama, Science Fiction]",8.1,2014,8.0817
6,Star Wars,"[Adventure, Action, Science Fiction]",8.1,1977,8.0702
7,Se7en,"[Crime, Mystery, Thriller]",8.1,1995,8.0658
8,The Silence of the Lambs,"[Crime, Drama, Thriller]",8.1,1991,8.0558
9,Memento,"[Mystery, Thriller]",8.1,2000,8.0514


### Strategy 4 — Hybrid (history + genres, weighted blend)

This is the **primary production strategy** in CineMatch.  
It blends content similarity (from watch history) with genre preference, and applies the Bayesian score as a quality dampener.

```
hybrid_score = (w_content × content_score + w_genre × genre_score) × bayesian_norm
```

In [15]:
def recommend_hybrid(watched_titles, genre_list, top_n=10,
                     w_content=0.7, w_genre=0.3):
    """Blend watch-history similarity with genre preference."""

    # ── Content scores (from history) ────────────────────────────────────────
    valid_idxs = [title_to_idx[t] for t in watched_titles if t in title_to_idx]
    content_scores = np.zeros(len(df), dtype=np.float64)
    if valid_idxs:
        for idx in valid_idxs:
            content_scores += cosine_sim[idx]
        content_scores /= len(valid_idxs)
        if content_scores.max() > 0:
            content_scores /= content_scores.max()   # normalise 0-1

    # ── Genre scores ─────────────────────────────────────────────────────────
    target = {g.strip().title() for g in genre_list}
    def genre_score(g_list):
        if not isinstance(g_list, list) or not target:
            return 0.0
        return min(sum(1 for g in g_list if g in target) / len(target), 1.0)

    genre_scores = df["genres_list"].apply(genre_score).values

    # ── Bayesian quality multiplier ───────────────────────────────────────────
    bayes = df.apply(bayesian_score, axis=1).values.astype(np.float64)
    bayes /= bayes.max()   # normalise 0-1

    # ── Blend ─────────────────────────────────────────────────────────────────
    total_w = w_content + w_genre
    w_content /= total_w
    w_genre   /= total_w
    hybrid = (w_content * content_scores + w_genre * genre_scores) * bayes

    watched_set = set(valid_idxs)
    ranked = sorted(enumerate(hybrid), key=lambda x: x[1], reverse=True)

    results = []
    for movie_idx, score in ranked:
        if movie_idx in watched_set:
            continue
        results.append(fmt(df.iloc[movie_idx], score=score, score_key="hybrid_score"))
        if len(results) >= top_n:
            break
    return results


# ── Demo ──────────────────────────────────────────────────────────────────────
history = ["The Dark Knight", "Inception"]
genres  = ["Action", "Crime"]
print(f"Hybrid: history={history}, genres={genres}")
show(recommend_hybrid(history, genres))

Hybrid: history=['The Dark Knight', 'Inception'], genres=['Action', 'Crime']


,title,genres,vote_avg,year,hybrid_score
0,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]",7.6,2012,0.5973
1,Batman Begins,"[Action, Crime, Drama]",7.5,2005,0.5310
2,Amidst the Devil's Wings,"[Drama, Action, Crime]",0.0,2014,0.3708
3,Kick-Ass,"[Action, Crime]",7.1,2010,0.3697
4,Sherlock Holmes: A Game of Shadows,"[Adventure, Action, Crime, Mystery]",7.0,2011,0.3637
5,Chicago Overcoat,"[Action, Crime, Thriller]",6.1,2009,0.3574
6,The Fugitive,"[Adventure, Action, Thriller, Crime, Mystery]",7.2,1993,0.3547
7,Kill Bill: Vol. 2,"[Action, Crime, Thriller]",7.6,2004,0.3547
8,Drive,"[Drama, Action, Thriller, Crime]",7.4,2011,0.3531
9,Scarface,"[Action, Crime, Drama, Thriller]",8.0,1983,0.3527


---
## Part 9 — Search

The search function in CineMatch is purely **string-matching** (no vectors involved).  
It scores candidates by: exact match → prefix match → token overlap → popularity.

In [16]:
def search(query, limit=8):
    q = query.strip().lower()
    tokens = q.split()

    # Exact substring match first
    candidates = df[df["norm_title"].str.contains(q, na=False, regex=False)].copy()

    # Fall back to all-tokens match if nothing found
    if candidates.empty and tokens:
        candidates = df[
            df["norm_title"].apply(lambda t: all(tok in t for tok in tokens))
        ].copy()

    if candidates.empty:
        print(f"No results for '{query}'")
        return pd.DataFrame()

    # Scoring: exact > prefix > token hits > popularity
    candidates["_exact"]  = (candidates["norm_title"] == q).astype(int) * 5
    candidates["_prefix"] = candidates["norm_title"].str.startswith(q).astype(int) * 3
    candidates["_tokens"] = candidates["norm_title"].apply(
        lambda t: sum(tok in t for tok in tokens)
    )

    result = candidates.sort_values(
        by=["_exact", "_prefix", "_tokens", "popularity", "vote_average"],
        ascending=False,
    ).head(limit)

    return result[["title", "genres_list", "vote_average", "release_date"]].rename(
        columns={"genres_list": "genres", "vote_average": "rating", "release_date": "year"}
    )


# ── Demo ──────────────────────────────────────────────────────────────────────
search("spider")

,title,genres,rating,year
3084,Spider,"[Drama, Mystery, Thriller]",6.4,2002-12-13
5,Spider-Man 3,"[Fantasy, Action, Adventure]",5.9,2007-05-01
159,Spider-Man,"[Fantasy, Action]",6.8,2002-05-01
30,Spider-Man 2,"[Action, Adventure, Fantasy]",6.7,2004-06-25
20,The Amazing Spider-Man,"[Action, Adventure, Fantasy]",6.5,2012-06-27
38,The Amazing Spider-Man 2,"[Action, Adventure, Fantasy]",6.5,2014-04-16
336,The Spiderwick Chronicles,"[Adventure, Family, Fantasy]",6.3,2008-02-14
1699,Along Came a Spider,"[Crime, Mystery, Thriller, Action]",6.1,2001-04-06


---
## Part 10 — Strategy Selector (the `recommend_for_user` logic)

CineMatch automatically picks the right strategy based on what a user has:

In [17]:
def recommend_for_user(watched_titles=None, genre_list=None, top_n=10):
    """
    Mirror of RecommendationEngine.recommend_for_user() in recommender.py.
    Automatically selects the best strategy.
    """
    has_history = bool(watched_titles)
    has_genres  = bool(genre_list)

    if has_history and has_genres:
        strategy = "hybrid"
        reason   = "Blending your saved genres with your watch history."
        movies   = recommend_hybrid(watched_titles, genre_list, top_n=top_n)

    elif has_history:
        strategy = "content_based"
        reason   = "Based on the films already in your library."
        movies   = recommend_by_history(watched_titles, top_n=top_n)

    elif has_genres:
        strategy = "genre_based"
        reason   = "Curated from the genres you selected."
        movies   = recommend_by_genres(genre_list, top_n=top_n)

    else:
        strategy = "popular"
        reason   = "Popular, highly rated picks to help you get started."
        # Fallback: genre_based across all genres (= popular films)
        all_genres = sorted({g for gl in df["genres_list"] if isinstance(gl, list) for g in gl})
        movies = recommend_by_genres(all_genres[:5], top_n=top_n, min_rating=7.5)

    print(f"Strategy : {strategy}")
    print(f"Reason   : {reason}")
    return show(movies, top_n)


# ── Demo — new user, no data ──────────────────────────────────────────────────
print("=== Cold start (new user) ===")
recommend_for_user()

=== Cold start (new user) ===
Strategy : popular
Reason   : Popular, highly rated picks to help you get started.


,title,genres,vote_avg,year,bayesian_score
0,The Shawshank Redemption,"[Drama, Crime]",8.5,1994,8.4710
1,The Godfather,"[Drama, Crime]",8.4,1972,8.3615
2,Pulp Fiction,"[Thriller, Crime]",8.3,1994,8.2741
3,Spirited Away,"[Fantasy, Adventure, Animation, Family]",8.3,2001,8.2440
4,The Godfather: Part II,"[Drama, Crime]",8.3,1974,8.2358
5,The Dark Knight,"[Drama, Action, Crime, Thriller]",8.2,2008,8.1826
6,Forrest Gump,"[Comedy, Drama, Romance]",8.2,1994,8.1738
7,The Empire Strikes Back,"[Adventure, Action, Science Fiction]",8.2,1980,8.1648
8,The Green Mile,"[Fantasy, Drama, Crime]",8.2,1999,8.1492
9,GoodFellas,"[Drama, Crime]",8.2,1990,8.1347


In [18]:
# ── Demo — returning user with history + genres ───────────────────────────────
print("=== Returning user ===")
recommend_for_user(
    watched_titles=["The Dark Knight", "Inception"],
    genre_list=["Action", "Crime"],
    top_n=10,
)

=== Returning user ===
Strategy : hybrid
Reason   : Blending your saved genres with your watch history.


,title,genres,vote_avg,year,hybrid_score
0,The Dark Knight Rises,"[Action, Crime, Drama, Thriller]",7.6,2012,0.5973
1,Batman Begins,"[Action, Crime, Drama]",7.5,2005,0.5310
2,Amidst the Devil's Wings,"[Drama, Action, Crime]",0.0,2014,0.3708
3,Kick-Ass,"[Action, Crime]",7.1,2010,0.3697
4,Sherlock Holmes: A Game of Shadows,"[Adventure, Action, Crime, Mystery]",7.0,2011,0.3637
5,Chicago Overcoat,"[Action, Crime, Thriller]",6.1,2009,0.3574
6,The Fugitive,"[Adventure, Action, Thriller, Crime, Mystery]",7.2,1993,0.3547
7,Kill Bill: Vol. 2,"[Action, Crime, Thriller]",7.6,2004,0.3547
8,Drive,"[Drama, Action, Thriller, Crime]",7.4,2011,0.3531
9,Scarface,"[Action, Crime, Drama, Thriller]",8.0,1983,0.3527


---
## Part 11 — Save Model Artifacts

In production, preprocessing is a **one-time step**.  
We pickle the computed objects so the FastAPI server just loads them at startup — no recomputation needed.

In [19]:
# Final clean DataFrame
movies_clean = df[[
    "id", "title", "overview", "genres_list",
    "vote_average", "vote_count", "release_date",
    "runtime", "popularity", "tags"
]].copy()

# Save
movies_clean.to_pickle(MODEL_DIR / "movies.pkl")

with open(MODEL_DIR / "cosine_sim.pkl", "wb") as f:
    pickle.dump(cosine_sim, f)

with open(MODEL_DIR / "indices.pkl", "wb") as f:
    pickle.dump(title_to_idx, f)

with open(MODEL_DIR / "id_to_index.pkl", "wb") as f:
    pickle.dump(id_to_idx, f)

with open(MODEL_DIR / "tfidf_vectorizer.pkl", "wb") as f:
    pickle.dump(tfidf, f)

print("Artifacts saved to models/:")
for p in sorted(MODEL_DIR.iterdir()):
    print(f"  {p.name:30s}  {p.stat().st_size / 1024**2:.1f} MB")

Artifacts saved to models/:
  cosine_sim.pkl                  175.8 MB
  id_to_index.pkl                 0.1 MB
  indices.pkl                     0.1 MB
  movies.pkl                      3.7 MB
  tfidf_vectorizer.pkl            0.6 MB


---
## Summary

| Step | What we did | Corresponding file |
|------|-------------|--------------------|
| Load & merge | Combined movies + credits CSVs | `data_preprocessing.py` |
| Parse JSON cols | `ast.literal_eval` on genres/cast/crew | `data_preprocessing.py` |
| Feature engineering | Built a `tags` string per movie | `data_preprocessing.py` |
| TF-IDF | Vectorised tags into 15k-dim sparse matrix | `data_preprocessing.py` |
| Cosine similarity | Computed N×N pairwise similarity | `data_preprocessing.py` |
| Content-based rec | Top-N by single movie similarity | `recommender.py` → `recommend_by_history` |
| History-based rec | Average similarity profile | `recommender.py` → `recommend_by_history` |
| Genre-based rec | Bayesian-scored genre filter | `recommender.py` → `recommend_by_genres` |
| Hybrid rec | Weighted content + genre blend | `recommender.py` → `recommend_hybrid` |
| Strategy selector | Auto-picks strategy by user state | `recommender.py` → `recommend_for_user` |
| Persist artifacts | Pickle to `models/` for fast server startup | `data_preprocessing.py` |

The `main.py` FastAPI app loads these pickles once at boot and exposes them via REST endpoints authenticated with JWT tokens stored in SQLite via SQLAlchemy.

---
*CineMatch concept notebook — built to understand the recommender engine before reading the full application code.*